In [45]:
# ===== 팀 프로젝트 시작 셀 =====
import os, sys
from google.colab import userdata

TOKEN = userdata.get('GITHUB_TOKEN')
NAME = "sooyong"
REPO_URL = "github.com/DA15-Lab/team3-ping-analytics.git"
PROJECT_DIR = "/content/team3-ping-analytics"

if not os.path.exists(PROJECT_DIR):
    !git clone -q https://{TOKEN}@{REPO_URL}
    %cd {PROJECT_DIR}
else:
    %cd {PROJECT_DIR}
    !git pull -q

!git config user.email "{NAME}@example.com"
!git config user.name "{NAME}"

!pip install -q -r requirements.txt

sys.path.append("/content/team3-ping-analytics/src")
from preprocess import *

print(module_test())
print("✅ 준비 완료")

/content/team3-ping-analytics
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 104.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
✅ 모듈 설치 완료
✅ 준비 완료


In [ ]:
# ===== 작업 저장 셀 =====
%cd {PROJECT_DIR}

from google.colab import _message
import json

with open(f'notebooks/{NAME}_analysis.ipynb', 'w') as f:
    json.dump(_message.blocking_request('get_ipynb', timeout_sec=30)['ipynb'], f)

!pip freeze | grep -v "google-colab" > requirements.txt

!git add .
!git commit -m "작업한 내용 메시지"
!git pull
!git push

In [46]:
hello()

'works!'

In [47]:
# 구글 드라이브 연결

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [48]:
# 라이브러리 불러오기

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [49]:
# 데이터 불러오기

accounts_attendance = pd.read_csv('/content/drive/MyDrive/코드잇 3팀 고급 프로젝트/00_원본 csv/accounts_attendance.csv')
accounts_user = pd.read_csv('/content/drive/MyDrive/코드잇 3팀 고급 프로젝트/00_원본 csv/accounts_user.csv')
accounts_userquestionrecord = pd.read_csv('/content/drive/MyDrive/코드잇 3팀 고급 프로젝트/00_원본 csv/accounts_userquestionrecord.csv')

In [50]:
accounts_user.columns

Index(['id', 'is_superuser', 'is_staff', 'gender', 'point', 'friend_id_list',
       'is_push_on', 'created_at', 'block_user_id_list', 'hide_user_id_list',
       'ban_status', 'report_count', 'alarm_count', 'pending_chat',
       'pending_votes', 'group_id'],
      dtype='object')

In [51]:
accounts_user['id'].nunique()

677085

In [52]:
accounts_user['ban_status'].unique()

array(['N', 'NB', 'W', 'RB'], dtype=object)

## 무료 충전 관련 테이블 구조 확인

### 필요 컬럼 준비

In [ ]:
for df in [
    attendance_base,
    questionset_base,
    point_base,
]:
    df["user_id"] = pd.to_numeric(
        df["user_id"],
        errors="coerce",
    ).astype("Int64")

### 데이터형, 형태 정리

In [ ]:
questionset_base["opening_time"] = pd.to_datetime(
    questionset_base["opening_time"],
    errors="coerce"
)

questionset_base["created_at"] = pd.to_datetime(
    questionset_base["created_at"],
    errors="coerce"
)

questionset_base["status"] = (
    questionset_base["status"]
    .astype("string")
    .str.strip()
    .str.upper()
)

point_base["created_at"] = pd.to_datetime(
    point_base["created_at"],
    errors="coerce"
)

point_base["delta_point"] = pd.to_numeric(
    point_base["delta_point"],
    errors="coerce",
)

### 크기 확인

In [ ]:
source_validation = pd.DataFrame({
    "table": [
        "accounts_attendance",
        "polls_questionset",
        "accounts_pointhistory",
    ],

    "rows": [
        len(attendance_base),
        len(questionset_base),
        len(point_base),
    ],

    "unique_users": [
        attendance_base["user_id"].nunique(),
        questionset_base["user_id"].nunique(),
        point_base["user_id"].nunique(),
    ],

    "null_user_rows": [
        attendance_base["user_id"].isna().sum(),
        questionset_base["user_id"].isna().sum(),
        point_base["user_id"].isna().sum(),
    ],
}).set_index("table")

display(source_validation)

### 출석 날짜 / 질문 세트 리스트 형태 확인

In [ ]:
attendance_type_validation = (
    attendance_base["attendance_date_list"]
    .dropna()
    .map(lambda value: type(value).__name__)
    .value_counts()
    .to_frame("rows")
)

display(attendance_type_validation)

display(
    attendance_base[
        ["user_id", "attendance_date_list"]
    ].head(10)
)

In [ ]:
question_list_type_validation = (
    questionset_base["question_piece_id_list"]
    .dropna()
    .map(lambda value: type(value).__name__)
    .value_counts()
    .to_frame("rows")
)

display(question_list_type_validation)

display(
    questionset_base[
        [
            "user_id",
            "question_piece_id_list",
            "opening_time",
            "status",
            "created_at",
        ]
    ].head(10)
)

### 질문 세트의 분포 확인

In [ ]:
questionset_status_validation = (
    questionset_base["status"]
    .value_counts(dropna=False)
    .to_frame("rows")
)

questionset_status_validation["rate_pct"] = (
    questionset_status_validation["rows"]
    / len(questionset_base)
    * 100
)

display(
    questionset_status_validation.round(2)
)

### 200 포인트를 얻은 기록 확인

In [ ]:
positive_point_values = (
    point_base.loc[
        point_base["delta_point"] > 0,
        "delta_point",
    ]
    .value_counts(dropna=False)
    .head(20)
    .rename_axis("delta_point")
    .to_frame("rows")
)

positive_point_values["rate_among_increase_pct"] = (
    positive_point_values["rows"]
    / (point_base["delta_point"] > 0).sum()
    * 100
)

display(positive_point_values.round(2))

In [ ]:
free_200_candidates = point_base.loc[
    point_base["delta_point"].eq(200)
].copy()

free_200_validation = pd.Series({
    "plus_200_rows": len(free_200_candidates),

    "unique_plus_200_users": (
        free_200_candidates["user_id"].nunique()
    ),

    "first_plus_200_at": (
        free_200_candidates["created_at"].min()
    ),

    "last_plus_200_at": (
        free_200_candidates["created_at"].max()
    ),

    "linked_question_record_rows": (
        free_200_candidates[
            "user_question_record_id"
        ].notna().sum()
    ),

    "null_question_record_rows": (
        free_200_candidates[
            "user_question_record_id"
        ].isna().sum()
    ),
}, name="value").to_frame()

display(free_200_validation)

display(
    free_200_candidates
    .sort_values("created_at")
    .head(20)
)

In [ ]:
accounts_pointhistory.head()

In [ ]:
accounts_pointhistory.shape

In [ ]:
accounts_pointhistory.isna().sum()

In [ ]:
target_point_values = [200, 777, 1000, 4000]

payment_amount_check = (
    point_base.loc[
        point_base["delta_point"].isin(
            target_point_values
        )
    ]
    .groupby("delta_point")
    .agg(
        rows=("delta_point", "size"),
        unique_users=("user_id", "nunique"),
        linked_question_rows=(
            "user_question_record_id",
            lambda values: values.notna().sum(),
        ),
        first_at=("created_at", "min"),
        last_at=("created_at", "max"),
    )
    .reindex(
        target_point_values,
        fill_value=0,
    )
)

payment_amount_check["total_point"] = (
    payment_amount_check.index
    * payment_amount_check["rows"]
)

display(payment_amount_check)

In [ ]:
# 1. 질문 기록 ID가 있는 포인트 이력만 선택
point_linked = (
    accounts_pointhistory.loc[
        accounts_pointhistory["user_question_record_id"].notna(),
        ["id", "user_id", "delta_point", "user_question_record_id"],
    ]
    .rename(columns={
        "id": "point_history_id",
        "user_id": "point_user_id",
    })
    .copy()
)

# 2. 연결할 질문 기록 준비
question_lookup = (
    accounts_userquestionrecord[
        ["id", "user_id", "chosen_user_id"]
    ]
    .rename(columns={
        "id": "question_record_id",
        "user_id": "answer_user_id",
    })
    .copy()
)

# 3. 조인 키의 자료형 통일
point_linked["user_question_record_id"] = pd.to_numeric(
    point_linked["user_question_record_id"],
    errors="raise",
).astype("Int64")

question_lookup["question_record_id"] = pd.to_numeric(
    question_lookup["question_record_id"],
    errors="raise",
).astype("Int64")

# 4. 포인트 이력 → 질문 기록 조인
point_question_joined = point_linked.merge(
    question_lookup,
    left_on="user_question_record_id",
    right_on="question_record_id",
    how="left",
    validate="many_to_one",
    indicator="join_result",
)

# 5. 조인 성공 여부 확인
matched = point_question_joined["join_result"].eq("both")

print(f"조인 대상 포인트 행: {len(point_linked):,}")
print(f"조인 후 행 수: {len(point_question_joined):,}")
print(f"질문 기록 연결 성공: {matched.sum():,}")
print(f"질문 기록 연결 실패: {(~matched).sum():,}")
print(f"연결 성공률: {matched.mean() * 100:.2f}%")

display(point_question_joined.head(10))

In [ ]:
accounts_userquestionrecord.head()

In [ ]:
accounts_paymenthistory.head()

In [ ]:
accounts_paymenthistory.isna().sum()

In [ ]:
accounts_paymenthistory.shape

In [ ]:
accounts_paymenthistory['user_id'].nunique()

In [ ]:
product_buy = accounts_paymenthistory.groupby('productId')['user_id'].count()

product_buy

In [ ]:
import pandas as pd

# Figma에 안내된 상품 가격
price_map = {
    "heart.200": 900,
    "heart.777": 1900,
    "heart.1000": 2900,
    "heart.4000": 9900,
}

# 1. 상품별 결제 건수
payment_summary = (
    accounts_paymenthistory
    .groupby("productId", dropna=False, observed=True)
    .size()
    .reset_index(name="payment_count")
)

# 2. 상품당 하트 수와 가격
payment_summary["hearts_per_payment"] = (
    payment_summary["productId"]
    .astype("string")
    .str.extract(r"^heart\.(\d+)$", expand=False)
    .astype("int64")
)

payment_summary["price_krw"] = (
    payment_summary["productId"].map(price_map)
)

# 가격표에 없는 상품이 있으면 확인
if payment_summary["price_krw"].isna().any():
    raise ValueError("가격표에 없는 상품이 있습니다. productId를 확인하세요.")

# 3. 상품별 총 하트와 추정 결제액
payment_summary["total_hearts"] = (
    payment_summary["hearts_per_payment"]
    * payment_summary["payment_count"]
)

payment_summary["estimated_payment_krw"] = (
    payment_summary["price_krw"]
    * payment_summary["payment_count"]
)

payment_summary = payment_summary.sort_values("hearts_per_payment")

# 4. 전체 합계 행 추가
total_row = pd.DataFrame([{
    "productId": "전체 합계",
    "payment_count": payment_summary["payment_count"].sum(),
    "total_hearts": payment_summary["total_hearts"].sum(),
    "estimated_payment_krw": payment_summary["estimated_payment_krw"].sum(),
}])

result = pd.concat(
    [payment_summary, total_row],
    ignore_index=True,
)

# 5. 보기 편하게 컬럼 순서와 이름 정리
result = result[[
    "productId",
    "hearts_per_payment",
    "price_krw",
    "payment_count",
    "total_hearts",
    "estimated_payment_krw",
]]

result.columns = [
    "상품",
    "상품당 하트",
    "상품 가격(원)",
    "결제 건수",
    "총 구매 하트",
    "추정 결제액(원)",
]

display(
    result.style.format(
        {column: "{:,.0f}" for column in result.columns[1:]},
        na_rep="-",
    )
)

In [ ]:
# 원본은 수정하지 않고 필요한 컬럼만 복사
users = accounts_user[
    ["id", "created_at", "is_staff", "is_superuser"]
].copy()

points = accounts_pointhistory[
    ["id", "user_id", "user_question_record_id", "delta_point", "created_at"]
].copy()

questions = accounts_userquestionrecord[
    ["id", "user_id", "chosen_user_id"]
].copy()

payments = accounts_paymenthistory[
    ["id", "user_id", "productId", "created_at"]
].copy()

receipts = event_receipts[
    ["id", "user_id", "event_id", "plus_point", "created_at"]
].copy()

event_info = events[["id", "plus_point"]].copy()

# 날짜 변환
# 시간대 없는 원본은 모두 동일한 시간대라고 가정
for df in [users, points, payments, receipts]:
    df["created_at"] = pd.to_datetime(
        df["created_at"],
        format="mixed",
        errors="raise",
        utc=True,
    )

# 기본키 중복 확인
for df in [users, points, questions, payments, receipts, event_info]:
    assert df["id"].notna().all(), "ID 결측 확인 필요"
    assert df["id"].is_unique, "ID 중복 확인 필요"

# 내부 계정 제외
internal = (
    users["is_staff"].isin([True, 1, "1", "True"])
    | users["is_superuser"].isin([True, 1, "1", "True"])
)

users = (
    users.loc[~internal, ["id", "created_at"]]
    .rename(columns={"id": "user_id", "created_at": "signup_at"})
    .copy()
)

In [ ]:
# 포인트 기록 → 투표 기록 연결
question_points = points.loc[
    points["user_question_record_id"].notna()
].merge(
    questions.rename(
        columns={
            "id": "user_question_record_id",
            "user_id": "voter_user_id",
        }
    ),
    on="user_question_record_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

assert question_points["_merge"].eq("both").all(), "연결되지 않은 질문 기록 확인 필요"

# 포인트 소유자가 투표자인지, 선택받은 사람인지 확인
is_voter = question_points["user_id"].eq(
    question_points["voter_user_id"]
)
is_chosen = question_points["user_id"].eq(
    question_points["chosen_user_id"]
)

question_points["earn_source"] = np.select(
    [is_voter & is_chosen, is_voter, is_chosen],
    ["question_self_selected_role_unclear", "question_voter", "question_chosen"],
    default="unknown",
)

# 양수 지급이면서 질문 참여자에게 지급된 기록
question_free = (
    question_points.loc[
        question_points["delta_point"].gt(0)
        & (is_voter | is_chosen),
        ["id", "user_id", "created_at", "delta_point", "earn_source"],
    ]
    .rename(
        columns={
            "id": "source_record_id",
            "created_at": "earned_at",
            "delta_point": "free_point_amount",
        }
    )
    .copy()
)

question_free["source_table"] = "accounts_pointhistory"

# 이벤트 수령 → 이벤트 명세 연결
event_free = receipts.merge(
    event_info.rename(
        columns={"id": "event_id", "plus_point": "planned_plus_point"}
    ),
    on="event_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

assert event_free["_merge"].eq("both").all(), "이벤트 연결 확인 필요"
assert event_free["plus_point"].eq(event_free["planned_plus_point"]).all()
assert event_free["plus_point"].gt(0).all()

event_free = event_free.rename(
    columns={
        "id": "source_record_id",
        "created_at": "earned_at",
        "plus_point": "free_point_amount",
    }
)

event_free["earn_source"] = "event"
event_free["source_table"] = "event_receipts"

# 질문 지급과 이벤트 지급이 중복될 가능성 확인
match_keys = ["user_id", "earned_at", "free_point_amount"]

overlap = question_free.merge(
    event_free[match_keys].drop_duplicates(),
    on=match_keys,
    how="inner",
)

assert overlap.empty, "질문 지급과 이벤트 지급의 중복 후보 확인 필요"

# 지급 기록을 세로로 합치기
free_columns = [
    "source_table", "source_record_id", "user_id",
    "earned_at", "free_point_amount", "earn_source",
]

free_events = pd.concat(
    [question_free[free_columns], event_free[free_columns]],
    ignore_index=True,
)

assert not free_events.duplicated(
    ["source_table", "source_record_id"]
).any()

# 분석 대상 가입자만 남기기
free_events = free_events.merge(
    users,
    on="user_id",
    how="inner",
    validate="many_to_one",
)

assert free_events["earned_at"].ge(free_events["signup_at"]).all(), \
    "가입 이전 지급 기록 확인 필요"

# 사용자 한 명당 한 행으로 요약
free_user = free_events.groupby("user_id").agg(
    first_free_at=("earned_at", "min"),
    free_receipt_rows_observed=("source_record_id", "size"),
    free_point_total_observed=("free_point_amount", "sum"),
    free_sources=("earn_source", lambda x: "|".join(sorted(set(x)))),
)

In [ ]:
# 가입자 명부에 무료 지급 요약 붙이기
user_funnel = users.merge(
    free_user,
    on="user_id",
    how="left",
    validate="one_to_one",
)

# 분석 대상의 결제 기록
payment_valid = payments.merge(
    users,
    on="user_id",
    how="inner",
    validate="many_to_one",
)

assert payment_valid["created_at"].ge(payment_valid["signup_at"]).all(), \
    "가입 이전 결제 기록 확인 필요"

# 사용자별 최초 결제 시각
first_payment = (
    payment_valid.groupby("user_id")["created_at"]
    .min()
    .rename("first_payment_at")
)

user_funnel = user_funnel.merge(
    first_payment,
    on="user_id",
    how="left",
    validate="one_to_one",
)

# 각 결제 기록에 첫 무료 획득 시각 붙이기
payment_check = payment_valid.merge(
    free_user[["first_free_at"]],
    on="user_id",
    how="left",
    validate="many_to_one",
)

# 첫 무료 획득보다 뒤에 발생한 결제만 선택
first_payment_after_free = (
    payment_check.loc[
        payment_check["created_at"].gt(payment_check["first_free_at"])
    ]
    .groupby("user_id")["created_at"]
    .min()
    .rename("first_payment_after_free_at")
)

user_funnel = user_funnel.merge(
    first_payment_after_free,
    on="user_id",
    how="left",
    validate="one_to_one",
)

has_free = user_funnel["first_free_at"].notna()
has_after = user_funnel["first_payment_after_free_at"].notna()

# 같은 시각이면 선후관계를 단정하지 않음
same_time_users = payment_check.loc[
    payment_check["created_at"].eq(payment_check["first_free_at"]),
    "user_id",
]

user_funnel["has_confirmed_free"] = has_free

user_funnel["paid_before_free"] = (
    user_funnel["first_payment_at"]
    .lt(user_funnel["first_free_at"])
    .astype("boolean")
    .where(has_free)
)

user_funnel["paid_at_first_free_timestamp"] = (
    user_funnel["user_id"].isin(same_time_users)
    .astype("boolean")
    .where(has_free)
)

user_funnel["has_payment_after_free"] = (
    has_after.astype("boolean").where(has_free)
)

# 무료 획득 기록이 없으면 실제 미획득으로 단정하지 않기 위해 구분
user_funnel["payment_path"] = "free_not_observed"
user_funnel.loc[has_free, "payment_path"] = "no_payment_after_free_observed"

prior = user_funnel["paid_before_free"].fillna(False)
same = user_funnel["paid_at_first_free_timestamp"].fillna(False)

user_funnel.loc[
    has_after & ~prior & ~same, "payment_path"
] = "first_observed_payment_after_free"

user_funnel.loc[
    has_after & prior, "payment_path"
] = "repeat_payment_after_free"

user_funnel.loc[
    has_free & same & ~prior, "payment_path"
] = "same_time_first_payment_ambiguous"

# 관찰된 지급 건수/합계의 결측만 0으로 처리
for col in ["free_receipt_rows_observed", "free_point_total_observed"]:
    user_funnel[col] = user_funnel[col].fillna(0).astype("int64")

# 실제 추출 시각 대신 마지막 기록 시각을 사용한 임시 관찰기간
temporary_end = max(
    users["signup_at"].max(),
    points["created_at"].max(),
    payments["created_at"].max(),
    receipts["created_at"].max(),
)

user_funnel["followup_days"] = (
    temporary_end - user_funnel["first_free_at"]
).dt.total_seconds() / 86400

user_funnel["followup_is_provisional"] = True

# 사용자 중복 / 지급 건수 검증
assert user_funnel["user_id"].is_unique
assert len(user_funnel) == len(users)
assert user_funnel["free_receipt_rows_observed"].sum() == len(free_events)

# 전체 퍼널
funnel_summary = pd.DataFrame({
    "stage": [
        "가입 유저 — 관리자 추측 계정 제외",
        "무료 지급 확인 유저",
        "무료 지급 이후 결제 유저",
    ],
    "users": [
        len(user_funnel),
        int(has_free.sum()),
        int(has_after.sum()),
    ],
})

display(funnel_summary)

# 무료 획득자 내부의 결제 경로
display(
    user_funnel.loc[has_free, "payment_path"]
    .value_counts()
    .rename_axis("payment_path")
    .to_frame("users")
)

# 실제 연결표 미리보기
display(user_funnel.loc[has_free].head(10))

In [ ]:
import ast

def parse_list(value):
    # 이미 리스트라면 그대로 사용
    if isinstance(value, (list, tuple, np.ndarray)):
        return list(value)

    # 결측이나 빈 문자열이면 빈 리스트
    if pd.isna(value) or str(value).strip() == "":
        return []

    # 문자열 리스트 → 실제 리스트
    parsed = ast.literal_eval(str(value))
    assert isinstance(parsed, (list, tuple)), "리스트 형식 확인 필요"

    return list(parsed)


# 기존 결과는 그대로 보존
comparison_base = user_funnel.copy()

for column in [
    "signup_at",
    "first_free_at",
    "first_payment_after_free_at",
]:
    comparison_base[column] = pd.to_datetime(
        comparison_base[column],
        format="mixed",
        utc=True,
    )

comparison_users = comparison_base[
    ["user_id", "signup_at"]
].copy()

assert comparison_users["user_id"].is_unique

In [ ]:
attendance_rows = accounts_attendance[
    ["user_id", "attendance_date_list"]
].copy()

attendance_rows["attendance_date"] = (
    attendance_rows["attendance_date_list"].map(parse_list)
)

# 날짜 리스트를 한 날짜당 한 행으로 펼치기
attendance_rows = (
    attendance_rows[["user_id", "attendance_date"]]
    .explode("attendance_date")
    .dropna(subset=["attendance_date"])
)

attendance_rows["attendance_date"] = pd.to_datetime(
    attendance_rows["attendance_date"],
    format="mixed",
    errors="raise",
    utc=True,
).dt.normalize()

# 같은 사람의 같은 날 출석은 한 번만 계산
attendance_rows = (
    attendance_rows
    .drop_duplicates(["user_id", "attendance_date"])
    .merge(
        comparison_users,
        on="user_id",
        how="inner",
        validate="many_to_one",
    )
)

# 가입 날짜보다 이전인 출석 확인
before_signup_day = (
    attendance_rows["attendance_date"]
    < attendance_rows["signup_at"].dt.normalize()
)

print("가입일 이전 출석 제외:", int(before_signup_day.sum()))

# 사용자별 최초 출석일
attendance_first = (
    attendance_rows.loc[~before_signup_day]
    .groupby("user_id")["attendance_date"]
    .min()
    .reset_index()
    .merge(
        comparison_users,
        on="user_id",
        validate="one_to_one",
    )
)

# 가능한 가장 이른 시각
# 가입 당일 출석이면 가입 시각보다 앞서지는 않도록 처리
attendance_first["lower_at"] = attendance_first[
    ["attendance_date", "signup_at"]
].max(axis=1)

# 가능한 가장 늦은 시각: 해당 날짜의 끝
attendance_first["upper_at"] = (
    attendance_first["attendance_date"]
    + pd.Timedelta(days=1)
    - pd.Timedelta(nanoseconds=1)
)

attendance_first["source"] = "attendance_assumed"

print(f"출석 확인 유저: {len(attendance_first):,}명")

In [ ]:
sets = polls_questionset[
    ["id", "user_id", "question_piece_id_list"]
].copy()

assert sets["id"].notna().all()
assert sets["id"].is_unique

sets = sets.loc[
    sets["user_id"].isin(comparison_users["user_id"])
].copy()

sets["piece_ids"] = sets["question_piece_id_list"].map(parse_list)

# 서로 다른 질문 항목 10개로 구성된 세트인지 확인
sets["valid_10_pieces"] = sets["piece_ids"].map(
    lambda x: len(x) == 10 and len(set(x)) == 10
)

print(
    "10개 구성 조건에서 벗어난 세트:",
    int((~sets["valid_10_pieces"]).sum()),
)

set_pieces = (
    sets.loc[
        sets["valid_10_pieces"],
        ["id", "user_id", "piece_ids"],
    ]
    .explode("piece_ids")
    .rename(
        columns={
            "id": "questionset_id",
            "piece_ids": "question_piece_id",
        }
    )
)

set_pieces["question_piece_id"] = pd.to_numeric(
    set_pieces["question_piece_id"],
    errors="raise",
).astype("int64")

# 투표 기록 준비
votes = accounts_userquestionrecord[
    ["id", "user_id", "question_piece_id", "created_at"]
].copy()

assert votes["id"].notna().all()
assert votes["id"].is_unique

assert not votes.duplicated(
    ["user_id", "question_piece_id"]
).any(), "같은 사용자의 동일 질문 항목에 복수 투표 기록이 있는지 확인 필요"

votes["created_at"] = pd.to_datetime(
    votes["created_at"],
    format="mixed",
    errors="raise",
    utc=True,
)

assert votes["created_at"].notna().all()

# 사용자 + 질문 항목을 함께 기준으로 연결
piece_check = set_pieces.merge(
    votes.rename(
        columns={
            "id": "vote_id",
            "created_at": "voted_at",
        }
    ),
    on=["user_id", "question_piece_id"],
    how="left",
    validate="many_to_one",
)

# 세트별로 연결된 투표 수와 마지막 투표 시각 계산
set_check = (
    piece_check
    .groupby(["questionset_id", "user_id"])
    .agg(
        matched_votes=("vote_id", "count"),
        completed_at=("voted_at", "max"),
    )
    .reset_index()
)

# 해당 세트의 10개 항목에 모두 투표한 경우
completed_sets = set_check.loc[
    set_check["matched_votes"].eq(10)
].merge(
    comparison_users,
    on="user_id",
    how="inner",
    validate="many_to_one",
)

assert completed_sets["completed_at"].ge(
    completed_sets["signup_at"]
).all()

# 사용자별 첫 세트 완성 시각
bundle_first = (
    completed_sets.groupby("user_id")["completed_at"]
    .min()
    .reset_index()
)

bundle_first["lower_at"] = bundle_first["completed_at"]
bundle_first["upper_at"] = bundle_first["completed_at"]
bundle_first["source"] = "completed_set_assumed"

print(f"완성 세트: {len(completed_sets):,}개")
print(f"세트 완성 유저: {len(bundle_first):,}명")

In [ ]:
# 기존에 실제 지급이 확인된 사용자의 첫 지급 시각
confirmed_first = comparison_base.loc[
    comparison_base["has_confirmed_free"],
    ["user_id", "first_free_at"],
].copy()

confirmed_first["lower_at"] = confirmed_first["first_free_at"]
confirmed_first["upper_at"] = confirmed_first["first_free_at"]
confirmed_first["source"] = "confirmed_payout"

anchor_columns = [
    "user_id",
    "lower_at",
    "upper_at",
    "source",
]

# 세 경로를 세로로 합치기
anchor_candidates = pd.concat(
    [
        confirmed_first[anchor_columns],
        attendance_first[anchor_columns],
        bundle_first[anchor_columns],
    ],
    ignore_index=True,
)

# 사용자별 최초 확인/추정 시점
expanded_anchor = (
    anchor_candidates.groupby("user_id")
    .agg(
        first_free_lower_at=("lower_at", "min"),
        first_free_upper_at=("upper_at", "min"),
        evidence_sources=(
            "source",
            lambda x: "|".join(sorted(set(x))),
        ),
    )
    .reset_index()
)

assert expanded_anchor["first_free_lower_at"].le(
    expanded_anchor["first_free_upper_at"]
).all()

In [ ]:
comparison_payments = payment_valid[
    ["user_id", "created_at"]
].copy()

comparison_payments["created_at"] = pd.to_datetime(
    comparison_payments["created_at"],
    format="mixed",
    utc=True,
)

payment_order = comparison_payments.merge(
    expanded_anchor,
    on="user_id",
    how="inner",
    validate="many_to_one",
)

# 기준 시점보다 확실히 뒤에 발생한 결제
after_upper = (
    payment_order["created_at"]
    > payment_order["first_free_upper_at"]
)

# 가능한 시간 구간 안에 있어 선후관계가 불분명한 결제
inside_interval = (
    payment_order["created_at"].ge(
        payment_order["first_free_lower_at"]
    )
    & payment_order["created_at"].le(
        payment_order["first_free_upper_at"]
    )
)

after_users = payment_order.loc[
    after_upper, "user_id"
].unique()

uncertain_users = payment_order.loc[
    inside_interval, "user_id"
].unique()

# 확대 기준의 사용자별 연결표 — 기존 user_funnel과 별개
expanded_user_funnel = comparison_users.merge(
    expanded_anchor,
    on="user_id",
    how="left",
    validate="one_to_one",
)

expanded_user_funnel["has_free_confirmed_or_assumed"] = (
    expanded_user_funnel["first_free_lower_at"].notna()
)

expanded_user_funnel["has_payment_definitely_after_anchor"] = (
    expanded_user_funnel["user_id"].isin(after_users)
)

# 이후 결제가 별도로 확인된 사람은 미확정 인원에서 제외
expanded_user_funnel["payment_order_uncertain_only"] = (
    expanded_user_funnel["user_id"].isin(uncertain_users)
    & ~expanded_user_funnel["has_payment_definitely_after_anchor"]
)

In [ ]:
# 기존 지급 확인 기준
old_free = int(comparison_base["has_confirmed_free"].sum())

old_after = int(
    comparison_base["first_payment_after_free_at"].notna().sum()
)

old_uncertain = int(
    (
        comparison_base["paid_at_first_free_timestamp"].fillna(False)
        & comparison_base["first_payment_after_free_at"].isna()
    ).sum()
)

# 출석·세트 완성까지 포함한 기준
new_free = int(
    expanded_user_funnel["has_free_confirmed_or_assumed"].sum()
)

new_after = int(
    expanded_user_funnel["has_payment_definitely_after_anchor"].sum()
)

new_uncertain = int(
    expanded_user_funnel["payment_order_uncertain_only"].sum()
)

# 기본 검증
assert expanded_user_funnel["user_id"].is_unique
assert len(expanded_user_funnel) == len(comparison_base)
assert new_free >= old_free
assert new_after >= old_after
assert new_after + new_uncertain <= new_free


def rate(numerator, denominator):
    return numerator / denominator * 100 if denominator else np.nan


rate_label = "이후 결제 비율(%, 관찰기간 미통일)"

funnel_comparison = pd.DataFrame(
    {
        "지급 확인 기준": [
            len(comparison_base),
            old_free,
            old_after,
            old_uncertain,
            rate(old_after, old_free),
        ],
        "출석·세트 완성 포함(추정)": [
            len(comparison_base),
            new_free,
            new_after,
            new_uncertain,
            rate(new_after, new_free),
        ],
    },
    index=[
        "가입 유저 수",
        "무료 획득 확인/추정 유저 수",
        "첫 확인/추정 시점 이후 결제 유저 수",
        "선후 미확정 결제만 있는 유저 수",
        rate_label,
    ],
)

display(
    funnel_comparison.style
    .format("{:,.0f}")
    .format(
        "{:,.2f}",
        subset=pd.IndexSlice[[rate_label], :],
    )
)

In [ ]:
accounts_userquestionrecord.head()

In [ ]:
accounts_userquestionrecord['answer_status'].value_counts()

In [ ]:
accounts_userquestionrecord['opened_times'].value_counts()

In [ ]:
accounts_userquestionrecord['status'].value_counts()

In [ ]:
accounts_user.head()

In [ ]:
accounts_user['group_id'].nunique()

In [ ]:
accounts_attendance.head()

In [ ]:
accounts_userquestionrecord.head()

In [ ]:
t1 = accounts_userquestionrecord[(accounts_userquestionrecord['has_read'] == 0) & (accounts_userquestionrecord['opened_times'] != 0)]

t1

In [ ]:
accounts_userquestionrecord['opened_times'].value_counts()

In [ ]:
t1['chosen_user_id'].nunique()

In [ ]:
accounts_user['pending_votes'].sum()

In [ ]:
accounts_user['pending_votes'].count()

In [ ]:
57302054/677085